In [1]:
import setup_paths
import torch
from torchinfo import summary
from model import EarlyFusion3DCNN

In [2]:
# Initialize the model
model = EarlyFusion3DCNN(num_classes=3)

Summary of the model

In [3]:
# Pass the input size: (Batch Size, Channels, Depth, Height, Width)
summary(
    model, 
    input_size=(4, 3, 128, 128, 128), 
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=20,
    row_settings=["var_names"]
)

Layer (type (var_name))                  Input Shape          Output Shape         Param #              Trainable
EarlyFusion3DCNN (EarlyFusion3DCNN)      [4, 3, 128, 128, 128] [4, 3]               --                   True
├─ConvBlock3D (block1)                   [4, 3, 128, 128, 128] [4, 32, 64, 64, 64]  --                   True
│    └─Conv3d (conv)                     [4, 3, 128, 128, 128] [4, 32, 128, 128, 128] 2,624                True
│    └─BatchNorm3d (bn)                  [4, 32, 128, 128, 128] [4, 32, 128, 128, 128] 64                   True
│    └─ReLU (relu)                       [4, 32, 128, 128, 128] [4, 32, 128, 128, 128] --                   --
│    └─MaxPool3d (pool)                  [4, 32, 128, 128, 128] [4, 32, 64, 64, 64]  --                   --
├─ConvBlock3D (block2)                   [4, 32, 64, 64, 64]  [4, 64, 32, 32, 32]  --                   True
│    └─Conv3d (conv)                     [4, 32, 64, 64, 64]  [4, 64, 64, 64, 64]  55,360               True
│  

The raw batch of 4 preprocessed brains ([4, 3, 128, 128, 128]) takes up 100 MB of space.

The Forward/backward size exploded to 5.7 Gigabytes (5704 MB). This is because PyTorch saves every single intermediate 3D feature map from Block 1, Block 2, Block 3, and Block 4 so it can calculate the gradients backward during training.

When we add everything up (input, parameters, etc) the total GPU memory cost for a single batch is ~5.8 GB.